#### Visualizzazione Mesh + Segmentazione + Landmark (CPU-only)
Possibili step da testare:
- Caricare una mesh `.obj`
- Caricare la segmentazione GT `.json`
- Caricare i landmark GT `.json`
- Visualizzare la mesh con Open3D
- Colorare la mesh per dente
- Aggiungere landmark e determinare colore secondo la classe 
- Cambiare vista (frontale, laterale, dall’alto)
- Salvare screenshot


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import numpy as np
import open3d as o3d
import pymeshlab
import sys
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd().parent
print('PROJECT ROOT==>',PROJECT_ROOT)

c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
PROJECT ROOT==> c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans


#### Funzioni per caricare mesh, segmentazione e landmark

In [3]:
def load_unsorted_mesh(mesh_path: Path):
    mesh = o3d.io.read_triangle_mesh(str(mesh_path))
    mesh.compute_vertex_normals()
    return mesh
def load_mesh(mesh_path: Path):
    ms = pymeshlab.MeshSet()
    ms.load_new_mesh(str(mesh_path))

    vertices = ms.current_mesh().vertex_matrix()
    faces = ms.current_mesh().face_matrix()

    mesh = o3d.geometry.TriangleMesh(
        vertices=o3d.utility.Vector3dVector(vertices),
        triangles=o3d.utility.Vector3iVector(faces)
    )
    mesh.compute_vertex_normals()
    return mesh
def load_segmentation(seg_path: Path):
    with open(seg_path, "r") as f:
        data = json.load(f)
    return np.array(data["labels"])
def load_landmarks(kpt_path: Path):
    with open(kpt_path, "r") as f:
        data = json.load(f)
    coords = np.array([obj["coord"] for obj in data["objects"]])
    classes = [obj["class"] for obj in data["objects"]]
    return coords, classes
def color_mesh_by_labels(mesh, labels, FDI_palette):
    colors = np.array([FDI_palette.get(int(l), [255,255,255]) for l in labels]) / 255.0
    mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
    return mesh

def create_landmark_spheres(coords, classes, landmark_palette):
    spheres = []
    for coord, cls in zip(coords, classes):
        sphere = o3d.geometry.TriangleMesh.create_sphere(radius=0.4)
        sphere.translate(coord)

        #colore in base alla classe
        color = landmark_palette.get(cls, [1.0, 1.0, 1.0])
        sphere.paint_uniform_color(color)

        sphere.compute_vertex_normals()
        spheres.append(sphere)
    return spheres
def set_view(vis, view="front", zoom=0.05):
    vc = vis.get_view_control()

    # 2) Viste cliniche standard
    if view == "front":          # vestibolare
        vc.set_front([0, 0, -1])
        vc.set_up([0, 1, 0])

    elif view == "top":          # occlusale
        vc.set_front([0, -1, 0])
        vc.set_up([0, 0, 1])

    elif view == "bottom":       # linguale/palatale
        vc.set_front([0, 1, 0])
        vc.set_up([0, 0, -1])

    elif view == "left":         # laterale sinistra
        vc.set_front([-1, 0, 0])
        vc.set_up([0, 1, 0])

    elif view == "right":        # laterale destra
        vc.set_front([1, 0, 0])
        vc.set_up([0, 1, 0])

    # 3) Zoom
    vc.set_zoom(zoom)

    # 4) Lookat automatico → evita mesh fuori campo
    bbox = vis.get_render_option()
    try:
        # Open3D 0.17: usa bounding box della geometria
        geom = vis.get_geometry(0)
        bb = geom.get_axis_aligned_bounding_box()
        vc.set_lookat(bb.get_center())
    except:
        pass

def apply_tooth_config_view(vis):
    vc = vis.get_view_control()

    vc.set_front([0.0, 0.0, 1.0])
    vc.set_up([-0.095852611932817064, 0.99539553785701518, 0.0])
    vc.set_lookat([0.41309100779889363, 1.1439286130517776, -101.39684191000001])
    vc.set_zoom(0.009)

def save_screenshot(mesh, spheres, out_path, view="front", width=1600, height=1600):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=True, width=width, height=height)  

    vis.add_geometry(mesh)
    for s in spheres:
        vis.add_geometry(s)

    vis.poll_events()
    vis.update_renderer()

    #set_view(vis, view)
    # Second poll/update is REQUIRED for camera update
    vis.poll_events()
    vis.update_renderer()

    vis.capture_screen_image(str(out_path))
    vis.destroy_window()

    print(f"Screenshot salvato in: {out_path}")
def set_default_open3d_view(vis, mesh):
    vc = vis.get_view_control()

    # 1) Bounding box della mesh
    bb = mesh.get_axis_aligned_bounding_box()
    center = bb.get_center()

    # 2) Vista frontale "automatica" di Open3D
    vc.set_front([0, 0, 1])   # asse Z+
    vc.set_up([0, 1, 0])      # asse Y+

    # 3) Lookat = centro della mesh
    vc.set_lookat(center)

    # 4) Zoom proporzionale alla dimensione della mesh
    extent = bb.get_extent()
    max_dim = max(extent)
    zoom = 1.0 / (max_dim / 100.0)   # tuning empirico
    vc.set_zoom(zoom)


def visualize(mesh, spheres, view="front", width=1200, height=1200):
    """
    Visualizza mesh + landmark in una finestra Open3D interattiva,
    impostando la vista desiderata.

    Parametri:
    - mesh: TriangleMesh Open3D
    - spheres: lista di landmark (TriangleMesh)
    - view: "front", "top", "bottom", "left", "right"
    - width, height: dimensioni della finestra
    """

    vis = o3d.visualization.Visualizer()
    vis.create_window(width=width, height=height, visible=True)

    vis.add_geometry(mesh)
    for s in spheres:
        vis.add_geometry(s)

    vis.poll_events()
    vis.update_renderer()
    set_default_open3d_view(vis, mesh)
    # Imposta la vista usando la tua funzione
    #set_view(vis, view)
    #apply_tooth_config_view(vis)
    vis.run()
    vis.destroy_window()

LANDMARK_PALETTE = {
    "Mesial":        [1.0, 0.0, 0.0],   # rosso
    "Distal":        [0.0, 1.0, 0.0],   # verde
    "Cusp":          [0.0, 0.0, 1.0],   # blu
    "InnerPoint":    [1.0, 1.0, 0.0],   # giallo
    "OuterPoint":    [0.0, 1.0, 1.0],   # ciano
    "FacialPoint":    [1.0, 0.0, 1.0],   # magenta
}
FDI_palette = {
    #Quadrante 1 (blu)
    11:[31,119,180], 12:[40,140,200], 13:[55,160,220], 14:[70,180,240],
    15:[90,200,255], 16:[110,210,255], 17:[130,220,255], 18:[150,230,255],

    #Quadrante 2 (verde)
    21:[44,160,44], 22:[60,180,60], 23:[80,200,80], 24:[100,220,100],
    25:[120,240,120], 26:[140,255,140], 27:[160,255,160], 28:[180,255,180],

    #Quadrante 3 (arancione)
    31:[255,127,14], 32:[255,150,40], 33:[255,170,60], 34:[255,190,80],
    35:[255,210,100], 36:[255,230,120], 37:[255,240,140], 38:[255,250,160],

    #Quadrante 4 (viola)
    41:[148,103,189], 42:[168,120,200], 43:[188,140,210], 44:[208,160,220],
    45:[228,180,230], 46:[248,200,240], 47:[255,210,250], 48:[255,230,255],

    #Gingiva
    0:[100,100,100]
}
def check_alignment(mesh, labels):
    if len(mesh.vertices) != len(labels):
        print("Mesh e JSON NON sono allineati!")
        print(f"Vertices mesh: {len(mesh.vertices)}")
        print(f"Labels JSON:  {len(labels)}")
        return False
    print("Mesh e JSON allineati.")
    return True


#### controllo numero di labels uniche per landmark e task di segmentazione

In [4]:
root = Path("../dataset/toothinstancenet_input")

#Set per raccogliere tutte le classi uniche
unique_fdi_labels = set()
unique_landmark_classes = set()

for file in root.iterdir():
    fname = file.name

    #Segmentazione denti
    if fname.endswith("_seg.json"):
        labels = load_segmentation(file)
        unique_fdi_labels.update(np.unique(labels))

    #Landmark
    if fname.endswith("_kpt.json"):
        _, classes = load_landmarks(file)
        unique_landmark_classes.update(classes)
#Risultati
print("=== Classi FDI (denti) trovate, count: {} ===".format(len(unique_fdi_labels)))
print(sorted(unique_fdi_labels))

print("\n=== Classi Landmark trovate, count: {} ===".format(len(unique_landmark_classes)))
print(sorted(unique_landmark_classes))

=== Classi FDI (denti) trovate, count: 33 ===
[np.int64(0), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48)]

=== Classi Landmark trovate, count: 6 ===
['Cusp', 'Distal', 'FacialPoint', 'InnerPoint', 'Mesial', 'OuterPoint']


#### verifica funzioni di utilità per caricare mesh, segmentazioni e landmark, colorare mesh e landmark, cambiare vista

In [ ]:
root = Path("../dataset/toothinstancenet_input")

case = "AKBDPB4C"
arc = "upper"

mesh_path = root / f"{case}_{arc}.obj"
seg_path  = root / f"{case}_{arc}_seg.json"
kpt_path  = root / f"{case}_{arc}__kpt.json"

mesh = load_mesh(mesh_path)

labels = load_segmentation(seg_path)
coords, classes = load_landmarks(kpt_path)

#heck alignment
check_alignment(mesh, labels)

#Apply segmentation
mesh = color_mesh_by_labels(mesh, labels, FDI_palette)

#Apply landmark
spheres = create_landmark_spheres(coords, classes, LANDMARK_PALETTE)

#Visualize
visualize(mesh, spheres, view="front")
"""
visualize(mesh, spheres, view="top")
visualize(mesh, spheres, view="left")
visualize(mesh, spheres, view="right")
"""

# Save screenshots
out_dir = Path("../screenshots_test") / f"{case}_{arc}"
save_screenshot(mesh, spheres, out_dir / "front.png", view="front")
#save_screenshot(mesh, spheres, out_dir / "top.png", view="top")
#save_screenshot(mesh, spheres, out_dir / "left.png", view="left")
#save_screenshot(mesh, spheres, out_dir / "right.png", view="right")


Mesh e JSON allineati.
Screenshot salvato in: ..\screenshots_test\AKBDPB4C_upper\front.png
